# Financial model comparison — 同資料、同時序、同評分

## tl;dr

本 Notebook 用同一份版本化程式比較 **Ridge、HistGradientBoosting、causal TCN、市場引導跨股 attention**。
無私人輸入時使用明確標示的合成 smoke；有已鎖定的私人 Kaggle bundle 時執行比較；`report` 模式只讀已完成的 run。
結果由下方實際執行輸出產生；**smoke 成功不是金融訊號、模型升級或正式服務完成**。

**本次保存輸出：真實研究報表模式。** 已完成48 fits；validation選到`M3-2`，官方score為`0.000868847322`，最佳固定baseline為`0.000447093446`。這是已看過的2025探索fold比較，沒有重用2026保留期、沒有在這次render重新訓練，也不是已核實歷史vintage的正式模型或Kaggle遠端執行結果。

## Context & Methods

主要分數：`mean((2 * prediction - 1) * target_intraday_return)`，每個指定股票都預測，不使用 top-k。
同一目標日的股票放在同一側，train 早於 validation；前處理只 fit train。

### Key Assumptions

- 舊歷史若只有現在取得的版本，維持 `research_assumption`；不冒稱當時發布已核實。
- 2026 年 1–5 月已經看過，這裡不再次當 unseen holdout。
- R2 至多 12 設定 × 4 季度 = 48 fits。R1 特徵消融、更多 seed、融合需另列預算。
- 選擇只看 validation 的官方 score；未勝 baseline 也保留結果，不一直加試到贏。

### 1. 選擇執行模式

設定以環境變數傳入：`FINPRED_NOTEBOOK_MODE=demo|compare|report`、`FINPRED_RUN_DIR`、`FINPRED_BUNDLE_ROOT`。
`compare` 可另外指定 `FINPRED_TABLE`、`FINPRED_FEATURE_ROWS`、`FINPRED_CONFIG`。
只有 `FINPRED_QUICK_SMOKE=1` 才把真實輸入改成 4 fits 的工程 smoke；這不算完整比較。

In [1]:
import hashlib
import json
import os
import shutil
import stat
import sys
import tempfile
import zipfile
from collections import defaultdict
from datetime import date, timedelta
from pathlib import Path
from IPython.display import Markdown, display

EXPECTED_BUNDLE_MANIFEST_SHA256 = ""

def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

bundle_root = Path(os.environ["FINPRED_BUNDLE_ROOT"]) if os.environ.get("FINPRED_BUNDLE_ROOT") else None
if bundle_root is None and Path("/kaggle/input").is_dir():
    candidates = list(Path("/kaggle/input").rglob("bundle-manifest.json"))
    if len(candidates) > 1:
        raise ValueError("Multiple bundles found; set FINPRED_BUNDLE_ROOT explicitly")
    bundle_root = candidates[0].parent if candidates else None

mode = os.environ.get("FINPRED_NOTEBOOK_MODE", "compare" if bundle_root else "demo")
if mode not in {"demo", "compare", "report"}:
    raise ValueError("Mode must be demo, compare or report")
work_root = Path(os.environ.get("FINPRED_WORK_DIR") or tempfile.mkdtemp(prefix="finpred-notebook-"))
work_root.mkdir(parents=True, exist_ok=True)
run_root = Path(os.environ.get("FINPRED_RUN_DIR", str(work_root / "comparison")))
quick_smoke = mode == "demo" or os.environ.get("FINPRED_QUICK_SMOKE") == "1"
print(json.dumps({"mode": mode, "quick_smoke": quick_smoke, "raw_rows_displayed": False}))

{"mode": "report", "quick_smoke": false, "raw_rows_displayed": false}


### 2. 核對輸入與程式版本

Kaggle bundle 以 manifest SHA256 固定整份輸入，逐檔再核對 bytes／hash。
程式碼從已核對的 archive 載入；換新版 dataset 不會偷偷換掉這次研究資料。
本 Notebook 不讀課程密碼、不下載即時資料，也不向競賽提交。

In [2]:
project_root = None
if bundle_root is not None:
    manifest_path = bundle_root / "bundle-manifest.json"
    manifest_hash = digest(manifest_path)
    if EXPECTED_BUNDLE_MANIFEST_SHA256 and manifest_hash != EXPECTED_BUNDLE_MANIFEST_SHA256:
        raise ValueError("Mounted dataset differs from the manifest pinned in this kernel")
    manifest = json.loads(manifest_path.read_text())
    allowed = {"training-table.json", "feature-rows.json", "model-config.json", "finpred-code.zip"}
    if set(manifest["files"]) != allowed:
        raise ValueError("Unexpected bundle payload list")
    for name, expected in manifest["files"].items():
        path = bundle_root / name
        if path.is_symlink() or digest(path) != expected["sha256"] or path.stat().st_size != expected["bytes"]:
            raise ValueError("Bundle payload hash/size mismatch")
    project_root = work_root / "vendor"
    if project_root.exists() and any(project_root.iterdir()):
        raise ValueError("Vendor directory must be empty; use a fresh work directory")
    project_root.mkdir(exist_ok=True)
    with zipfile.ZipFile(bundle_root / "finpred-code.zip") as archive:
        entries = archive.infolist()
        if len({e.filename for e in entries}) != len(entries) or sum(e.file_size for e in entries) > 16 * 1024 * 1024:
            raise ValueError("Duplicate or oversized source archive")
        for entry in entries:
            relative = Path(entry.filename)
            if relative.is_absolute() or ".." in relative.parts or stat.S_ISLNK(entry.external_attr >> 16):
                raise ValueError("Unsafe code archive member")
            if relative.suffix != ".py" or not (entry.filename.startswith("src/finpred/") or entry.filename == "scripts/compare_architectures.py"):
                raise ValueError("Unexpected archive member")
            target = project_root / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(entry))
    table_path = bundle_root / "training-table.json"
    feature_path = bundle_root / "feature-rows.json"
    config_path = bundle_root / "model-config.json"
else:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        candidates = (candidate, candidate / "ml_financial_prediction")
        project_root = next((p for p in candidates if (p / "src/finpred").is_dir()), None)
        if project_root:
            break
    if project_root is None and mode != "report":
        raise ValueError("Run from the source checkout or supply a verified private bundle")
    table_path = Path(os.environ["FINPRED_TABLE"]) if os.environ.get("FINPRED_TABLE") else None
    feature_path = Path(os.environ["FINPRED_FEATURE_ROWS"]) if os.environ.get("FINPRED_FEATURE_ROWS") else None
    config_path = Path(os.environ.get("FINPRED_CONFIG", str(project_root / "configs/research/model_families.json"))) if project_root else None

if project_root:
    sys.path.insert(0, str(project_root / "src"))
print("Input/code version checks completed; no raw input rows displayed.")

Input/code version checks completed; no raw input rows displayed.


## Data

### 3. 讀取資料，或建立工程用合成資料

合成資料固定 seed／公式，50 股、80 個 weekday 位置；不代表真實交易日曆、歷史發布證據或金融效果。
真實比較使用獨立的完整 feature rows 與已完成 labels，不能為了有 label 就刪掉可見股票特徵。

In [3]:
if mode == "demo":
    days, current = [], date(2024, 10, 1)
    while len(days) < 80:
        if current.weekday() < 5:
            days.append(current.isoformat())
        current += timedelta(days=1)
    stock_ids = [str(1000 + i) for i in range(50)]
    rows, feature_rows = [], []
    for day_index, day in enumerate(days):
        for stock_index, stock in enumerate(stock_ids):
            signal = ((day_index + stock_index) % 7 - 3) / 3
            market_signal = (day_index % 5 - 2) / 2
            ret = 0.003 * signal + 0.001 * market_signal
            features = {"stock_id": stock, "target_date": day,
                        "feature_cutoff": day + "T08:10:00+08:00",
                        "feature_available_at": day + "T07:00:00+08:00",
                        "signal": signal, "market_signal": market_signal,
                        "sparse": None if day_index % 9 == 0 else float(stock_index % 3)}
            feature_rows.append(features)
            rows.append({**features, "label_available_at": day + "T23:59:00+08:00",
                         "y": int(ret > 0), "adj_open": 100.0, "adj_close": 100.0 * (1 + ret),
                         "target_intraday_return": ret})
    table = {"schema_version": "training-table-v1", "readiness": "synthetic",
             "feature_version": "notebook-demo-v1", "feature_columns": ["signal", "market_signal", "sparse"],
             "availability_evidence": "Synthetic generated examples, no real market availability claim.",
             "rows": rows}
    table_path, feature_path = work_root / "demo-table.json", work_root / "demo-features.json"
    table_path.write_text(json.dumps(table))
    feature_path.write_text(json.dumps(feature_rows))
elif mode == "compare":
    if table_path is None or feature_path is None:
        raise ValueError("Comparison requires the exact table and separate feature rows")
    table = json.loads(table_path.read_bytes())
    feature_rows = json.loads(feature_path.read_bytes())

if mode != "report":
    print(json.dumps({"readiness": table["readiness"], "completed_label_rows": len(table["rows"]),
                      "feature_count": len(table["feature_columns"]),
                      "input_sha256_prefix": digest(table_path)[:16]}))

## Results

### 4. 執行同一比較介面，或讀取已完成結果

模型實作與評分只在 `finpred.training.architecture_comparison` 維護一份。
輸出目錄不可覆寫；重跑要用新目錄，讀舊結果則使用 `report` 模式。

In [4]:
if mode != "report":
    from finpred.training.architecture_comparison import compare_architectures
    report = compare_architectures(
        table, run_root, feature_rows=feature_rows, config=json.loads(config_path.read_text()),
        allow_research=table["readiness"] == "research_assumption", quick_smoke=quick_smoke,
        loaded_input_bytes=table_path.read_bytes(),
    )
else:
    report = json.loads((run_root / "report.json").read_text())

if report.get("schema_version") != "architecture-comparison-v1":
    raise ValueError("Expected a completed four-architecture comparison report")
if report.get("holdout_evaluated") is not False or report.get("production_eligible") is not False:
    raise ValueError("This research notebook must not claim production or revisit the spent holdout")
if not report.get("results"):
    raise ValueError("No fitted model results; do not write a success conclusion")
results = report["results"]
keys = {(r["fold"], r["config_id"]) for r in results}
if len(keys) != len(results) or report["completed_fit_count"] != len(results):
    raise ValueError("Duplicate or incomplete fit bookkeeping")
if {r["model_id"] for r in results} != {"M1", "M2", "M3", "M4"}:
    raise ValueError("All four architecture families must be represented")
if not report["quick_smoke_not_full_research"] and len(results) != 48:
    raise ValueError("Full R2 research requires exactly 48 registered fits")
fold_counts = defaultdict(set)
for result in results:
    fold_counts[result["fold"]].add(result["count"])
if any(len(counts) != 1 for counts in fold_counts.values()):
    raise ValueError("Models do not use the same scored row counts in each fold")
print(json.dumps({"completed_fit_count": report["completed_fit_count"],
                  "quick_smoke_not_full_research": report["quick_smoke_not_full_research"],
                  "holdout_evaluated": report["holdout_evaluated"],
                  "production_eligible": report["production_eligible"],
                  "input_payload_sha256": report.get("input_payload_sha256"),
                  "source_report_sha256": digest(run_root / "report.json")}))

{"completed_fit_count": 48, "quick_smoke_not_full_research": false, "holdout_evaluated": false, "production_eligible": false, "input_payload_sha256": "5bef3e7b71d926a83873c4135fae00ce9575bb9dd74ec6a622116266d2cb5855", "source_report_sha256": "2326799f8f16c476d6885e0d1bac1c4509d643bd022365da4d90e4697e6917ad"}


### 5. 依相同 validation 鍵比較

表格依各 fold 樣本數加權，使用官方 score。只展示設定摘要，不把逐股預測或私人資料貼入 Notebook。

In [5]:
aggregated = defaultdict(lambda: {"weighted_score": 0.0, "count": 0, "folds": 0, "fit_seconds": 0.0})
for result in report["results"]:
    key = (result["model_id"], result["config_id"])
    entry = aggregated[key]
    entry["weighted_score"] += float(result["score"]) * int(result["count"])
    entry["count"] += int(result["count"])
    entry["folds"] += 1
    entry["fit_seconds"] += float(result["fit_seconds"])
summary = [{"model": model, "configuration": config_id,
            "score": values["weighted_score"] / values["count"], **values}
           for (model, config_id), values in aggregated.items()]
for value in summary:
    recorded = report["configuration_scores"][value["configuration"]]
    if abs(value["score"] - recorded) > 1e-12:
        raise ValueError("Displayed aggregation disagrees with the saved report")
summary.sort(key=lambda value: (-value["score"], value["configuration"]))
lines = ["| Model | Configuration | Validation score | Folds | Fit seconds |",
         "|---|---|---:|---:|---:|"]
for value in summary[:12]:
    lines.append(f"| {value['model']} | {value['configuration']} | {value['score']:.10f} | {value['folds']} | {value['fit_seconds']:.2f} |")
display(Markdown("\n".join(lines)))
print("Validation-selected configuration:", json.dumps(report.get("validation_selected_configuration"), ensure_ascii=False))
print("Declared baseline scores:", json.dumps(report.get("baseline_scores"), sort_keys=True))
print("Beats declared baseline:", report.get("beats_baseline"))

| Model | Configuration | Validation score | Folds | Fit seconds |
|---|---|---:|---:|---:|
| M3 | M3-2 | 0.0008688473 | 4 | 478.18 |
| M4 | M4-1 | 0.0008672120 | 4 | 146.92 |
| M3 | M3-3 | 0.0008305537 | 4 | 516.58 |
| M4 | M4-2 | 0.0008119589 | 4 | 154.76 |
| M3 | M3-1 | 0.0007913883 | 4 | 319.39 |
| M4 | M4-3 | 0.0007382094 | 4 | 154.80 |
| M2 | M2-3 | 0.0005699374 | 4 | 5.93 |
| M2 | M2-1 | 0.0005207133 | 4 | 4.67 |
| M1 | M1-2 | 0.0004844244 | 4 | 0.06 |
| M1 | M1-1 | 0.0004831169 | 4 | 0.06 |
| M1 | M1-3 | 0.0004802322 | 4 | 0.06 |
| M2 | M2-2 | 0.0003840671 | 4 | 5.12 |

Validation-selected configuration: {"config_id": "M3-2", "model_id": "M3", "dimension": "dropout", "value": 0.1, "fixed_design": {"lookback": 20, "hidden_channels": 32, "kernel_size": 3, "dilations": [1, 2, 4], "blocks": 3, "convolutions_per_block": 2, "padding": "left_only_with_mask", "epochs": 20, "primary_head": "raw_return_regression", "lookback_unit": "target_market_sessions_with_missing_positions", "include_prediction_cutoff_vector": true}}
Declared baseline scores: {"always_0": 0.00044709344600899636, "always_1": -0.00044709344600899636, "train_majority": 0.00044709344600899636}
Beats declared baseline: True


## Takeaways

### 6. 用實際結果決定下一步

以下結論只描述本次執行。工程 smoke、金融樣本外增益、每日運作是三種不同驗收，不互相替代。

In [6]:
if report["quick_smoke_not_full_research"]:
    display(Markdown(f"**工程 smoke 完成 {report['completed_fit_count']} fits。** "
                     "這是執行路徑驗證，不能選正式模型、宣稱金融增益或算作完整48-fit研究。"))
else:
    display(Markdown(f"已完成 **{report['completed_fit_count']} fits** 的研究比較。"
                     f"Validation baseline 勝出狀態：**{report.get('beats_baseline')}**。"
                     "保留負結果；來源／時點政策與後續未見session仍須獨立驗收。"))
display(Markdown("Kaggle可以在同一Notebook比較多個模型；建議一個版本化比較介面，"
                 "研究選定recipe後，日常只刷新該recipe。Kaggle排隊與quota沒有09:00前完成保證。"))

已完成 **48 fits** 的研究比較。Validation baseline 勝出狀態：**True**。保留負結果；來源／時點政策與後續未見session仍須獨立驗收。

Kaggle可以在同一Notebook比較多個模型；建議一個版本化比較介面，研究選定recipe後，日常只刷新該recipe。Kaggle排隊與quota沒有09:00前完成保證。

來源與操作：[Kaggle 官方 CLI](https://github.com/Kaggle/kaggle-cli)、[kernel metadata](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels_metadata.md)。
私人 bundle 與帳戶設定見專案 `docs/KAGGLE_TRAINING.md`。Notebook 不含API token；課程密碼不需要帶進訓練kernel。